In [ ]:
import pandas as pd, json, os, re, zipfile, pathlib
excel_path = r"C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP\ListOfNames.xlsx"
site_dir   = r"C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP"
os.makedirs(site_dir, exist_ok=True)

googlesheet = 'https://docs.google.com/spreadsheets/d/19IE4FLYkJ1MAvawhhZx5WEnj0xowtOIoqPIB-E-6rSU/edit?usp=sharing'
sheet_id = "19IE4FLYkJ1MAvawhhZx5WEnj0xowtOIoqPIB-E-6rSU"
gid = 0  # change if your tab isn't the first one
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df = pd.read_csv(url)
df.head()


In [7]:
"""
Rename images inside subfolders that match the pattern: number-number (e.g., 123-456).

For each image, OCR the text, extract:
  - Firstname: ...
  - Lastname:  ...
Then rename the image to:
  Firstname Lastname.<original_extension>

Requirements:
  pip install pytesseract pillow opencv-python
  Install Tesseract OCR:
    - Windows: https://github.com/UB-Mannheim/tesseract/wiki
  Then set TESSERACT_CMD below if needed.
"""

from __future__ import annotations

import os
import re
import sys
import shutil
from pathlib import Path
from typing import Optional, Tuple

import cv2
import pytesseract

# --- CONFIG ---
ROOT_DIR = Path(r"C:\Users\sanaz\Downloads\Final_Craft-20260222T225357Z-1-001\Final_Craft")

# If pytesseract can't find tesseract.exe, uncomment and set this:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Folder name must look like "123-456"
FOLDER_RE = re.compile(r"^\d+-\d+$")

# Image extensions to process
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff", ".bmp"}

# --- OCR + PARSING HELPERS ---

def preprocess_for_ocr(img_bgr):
    """Improve OCR readability (no guarantee; depends on your images)."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    # denoise
    gray = cv2.bilateralFilter(gray, 9, 75, 75)
    # increase contrast (adaptive)
    thr = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 10
    )
    return thr

def ocr_text(image_path: Path) -> str:
    img = cv2.imread(str(image_path))
    if img is None:
        return ""
    proc = preprocess_for_ocr(img)

    # Try two OCR passes: processed + original gray
    txt1 = pytesseract.image_to_string(proc, lang="eng")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    txt2 = pytesseract.image_to_string(gray, lang="eng")

    # Combine
    return (txt1 + "\n" + txt2).strip()

def clean_name(s: str) -> str:
    # Keep letters, spaces, apostrophes, hyphens (for names like O'Neil, Anne-Marie)
    s = s.strip()
    s = re.sub(r"[\r\n\t]+", " ", s)
    s = re.sub(r"\s{2,}", " ", s)
    s = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿ'\- ]+", "", s)  # includes accented letters
    return s.strip()

def extract_first_last(text: str) -> Tuple[Optional[str], Optional[str]]:
    """
    Extracts Firstname and Lastname from OCR text.
    Handles variations like:
      Firstname: John
      First name : John
      Lastname: Doe
      Last name: Doe
    """
    t = text

    # Normalize common OCR weirdness
    t = t.replace("ﬁ", "fi").replace("ﬂ", "fl")
    t = re.sub(r"[ ]{2,}", " ", t)

    first_patterns = [
        r"First\s*name\s*[:\-]\s*([A-Za-zÀ-ÖØ-öø-ÿ'\- ]{1,50})",
        r"Firstname\s*[:\-]\s*([A-Za-zÀ-ÖØ-öø-ÿ'\- ]{1,50})",
        r"First\s*[:\-]\s*([A-Za-zÀ-ÖØ-öø-ÿ'\- ]{1,50})",
    ]
    last_patterns = [
        r"Last\s*name\s*[:\-]\s*([A-Za-zÀ-ÖØ-öø-ÿ'\- ]{1,50})",
        r"Lastname\s*[:\-]\s*([A-Za-zÀ-ÖØ-öø-ÿ'\- ]{1,50})",
        r"Last\s*[:\-]\s*([A-Za-zÀ-ÖØ-öø-ÿ'\- ]{1,50})",
        r"Surname\s*[:\-]\s*([A-Za-zÀ-ÖØ-öø-ÿ'\- ]{1,50})",
        r"Family\s*name\s*[:\-]\s*([A-Za-zÀ-ÖØ-öø-ÿ'\- ]{1,50})",
    ]

    first = None
    last = None

    for p in first_patterns:
        m = re.search(p, t, flags=re.IGNORECASE)
        if m:
            first = clean_name(m.group(1))
            break

    for p in last_patterns:
        m = re.search(p, t, flags=re.IGNORECASE)
        if m:
            last = clean_name(m.group(1))
            break

    # Sometimes OCR captures extra words after name; take first 1-3 tokens
    def trim_tokens(name: Optional[str]) -> Optional[str]:
        if not name:
            return None
        toks = name.split()
        toks = toks[:3]
        return " ".join(toks).strip()

    return trim_tokens(first), trim_tokens(last)

def safe_target_name(folder: Path, first: str, last: str, ext: str, src: Path) -> Path:
    base = f"{first} {last}".strip()
    base = re.sub(r"\s{2,}", " ", base)
    # Windows filename safety
    base = re.sub(r'[<>:"/\\|?*]+', "", base).strip().rstrip(".")
    if not base:
        base = src.stem
    candidate = folder / f"{base}{ext.lower()}"
    if not candidate.exists():
        return candidate
    # If already exists, add a counter
    for i in range(2, 9999):
        candidate = folder / f"{base} ({i}){ext.lower()}"
        if not candidate.exists():
            return candidate

    # Worst case fallback
    return folder / f"{base} ({os.getpid()}){ext.lower()}"

def main():
    if not ROOT_DIR.exists():
        print(f"ROOT_DIR not found: {ROOT_DIR}")
        sys.exit(1)

    renamed = 0
    skipped = 0
    failed = 0

    for sub in ROOT_DIR.iterdir():
        if not sub.is_dir():
            continue
        if not FOLDER_RE.match(sub.name):
            continue
        for img_path in sub.rglob("*"):
            if not img_path.is_file():
                continue
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            text = ocr_text(img_path)
            first, last = extract_first_last(text)

            if not first or not last:
                print(f"[SKIP] Could not find Firstname/Lastname in: {img_path}")
                skipped += 1
                continue

            target = safe_target_name(img_path.parent, first, last, img_path.suffix, img_path)

            # If the filename is already correct, skip
            if img_path.resolve() == target.resolve():
                skipped += 1
                continue

            try:
                img_path.rename(target)
                print(f"[OK] {img_path.name} -> {target.name}")
                renamed += 1
            except Exception as e:
                print(f"[FAIL] {img_path} : {e}")
                failed += 1

    print("\n--- Summary ---")
    print(f"Renamed: {renamed}")
    print(f"Skipped: {skipped}")
    print(f"Failed : {failed}")

if __name__ == "__main__":
    main()


--- Summary ---
Renamed: 0
Skipped: 0
Failed : 0


In [ ]:
import shutil
from pathlib import Path

def copy_with_increment(src, dest_folder):
    dest_folder = Path(dest_folder)
    dest_folder.mkdir(parents=True, exist_ok=True)
    src_path = Path(src)
    dest_path = dest_folder / src_path.name
    if dest_path.exists():
        stem = dest_path.stem
        suffix = dest_path.suffix
        counter = 2
        while (dest_folder / f"{stem}_{counter}{suffix}").exists():
            counter += 1
        dest_path = dest_folder / f"{stem}_{counter}{suffix}"
    shutil.copy(src_path, dest_path)
    print(f"Copied to: {dest_path}")

# Example usage:
# copy_with_increment("C:/path/to/file.jpg", "C:/path/to/destination")